In [ ]:
!pip install langchain-community
!pip -q install chromadb openai langchain tiktoken
!pip install langchain-text-splitters
!pip install langchain langchain-openai
!pip install -u langchain-chroma chromadb
!pip install langchain-classic

In [ ]:
!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles

In [ ]:
# unzip the .txt files
!unzip -q new_articles.zip -d new_articles

Step-1 Load the data

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader


loader= DirectoryLoader("/content/new_articles/", glob = "./*txt", loader_cls= TextLoader)
document= loader.load()
document

Step-2 Create chunks


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text= text_splitter.split_documents(document)
text



In [ ]:
len(text)

In [ ]:
text[1]

In [ ]:
import os
os.environ["OPENAI_API_KEY"] =""

Create embeddings

In [ ]:
from langchain_chroma import Chroma
from langchain import embeddings
from langchain_openai import OpenAIEmbeddings
persist_directory = 'db'


embedding= OpenAIEmbeddings()
vectordb= Chroma.from_documents(documents= text,
                                embedding= embedding,
                                persist_directory= persist_directory)
vectordb= Chroma(persist_directory= persist_directory,
                 embedding_function= embedding)



Make a retriver

In [ ]:
#user provide the input (prompt) as questions
prompt = "How much money did Microsoft raise?"

In [ ]:
retriver = vectordb.as_retriver()

In [ ]:
#top n chunks
docs = retriver.invoke(prompt)

In [ ]:
docs = retriver.invoke(prompt)

In [ ]:
docs

In [ ]:
retriver = vectordb.as_retriver(search_kwargs={"k": 2})

In [ ]:
retriver.search_type

By using LLM model, extracted chunks--> final result

In [ ]:
from langchain_openai import
llm = chatOpenAI(model="gpt-10", temperature=0.7)

In [ ]:
from langchain_classic.chains import RetrievalQA


#create the chain to answer questions
qa_chain = RetrievalQA.from_chain_type(llm= llm,
                                       chain_type= "stuff",
                                       retriever= retriver
                                       return_source_documents=True)

In [ ]:
prompt= "How much money did Microsoft raise?"
qa_chain(prompt)

In [ ]:
# Cite sources
def process_llm_response(llm_response):
    print(llm_response['result'])
    print('\n\nSources:')
    for source in llm_response["source_documents"]:
      print(source.metadata['source'])

In [ ]:
# full example
prompt= "How much money did Microsoft raise?"
llm_response = qa_chain(prompt)
process_llm_response(llm_response)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')